In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Device: cuda
GPU Memory: 6.0 GB


In [2]:
train_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/train_final.csv')
val_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/val_final.csv')
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')

train_df['text'] = train_df['headline'].fillna('') + ' ' + train_df['content'].fillna('')
val_df['text'] = val_df['headline'].fillna('') + ' ' + val_df['content'].fillna('')
test_df['text'] = test_df['headline'].fillna('') + ' ' + test_df['content'].fillna('')

label2id = {'authentic': 0, 'fake': 1, 'ai_fake': 2}
id2label = {0: 'authentic', 1: 'fake', 2: 'ai_fake'}

train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)
test_df['label_id'] = test_df['label'].map(label2id)

train_df = train_df.dropna(subset=['text', 'label_id'])
val_df = val_df.dropna(subset=['text', 'label_id'])
test_df = test_df.dropna(subset=['text', 'label_id'])

print("Data loaded!")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Data loaded!
Train: 10500, Val: 2250, Test: 2250


In [3]:
print("Loading BLOOM tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bigscience/bloom-560m")
tokenizer.pad_token = tokenizer.eos_token
print(" Tokenizer loaded!")

class BanglaNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = BanglaNewsDataset(train_df['text'].values, train_df['label_id'].values, tokenizer)
val_dataset = BanglaNewsDataset(val_df['text'].values, val_df['label_id'].values, tokenizer)
test_dataset = BanglaNewsDataset(test_df['text'].values, test_df['label_id'].values, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

print(" Datasets ready!")
print(f"Train batches: {len(train_loader)}")

Loading BLOOM tokenizer...
 Tokenizer loaded!
 Datasets ready!
Train batches: 1313


In [4]:
print("Loading BLOOM model...")

model = AutoModelForSequenceClassification.from_pretrained(
    "bigscience/bloom-560m",
    num_labels=3,
    torch_dtype=torch.float32
)
model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_key_value"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(" BLOOM LoRA model ready!")

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=1e-5)
total_steps = len(train_loader) * 3
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)
print(f"Total training steps: {total_steps}")

Loading BLOOM model...


Some weights of BloomForSequenceClassification were not initialized from the model checkpoint at bigscience/bloom-560m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 789,504 || all params: 560,007,168 || trainable%: 0.1410
 BLOOM LoRA model ready!
Total training steps: 3939


In [5]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = outputs.logits.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return preds, true_labels

print("Training started!")
best_f1 = 0

for epoch in range(3):
    print(f"\nEpoch {epoch+1}/3")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    print(f"Train Loss: {train_loss:.4f}")
    val_preds, val_true = evaluate(model, val_loader)
    val_f1 = f1_score(val_true, val_preds, average='macro')
    print(f"Val Macro F1: {val_f1*100:.2f}%")
    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('C:/Users/Riyad/projects/fake_news/bloom_lora_best_model')
        print(f" Best model saved!")

print(f"\nBest Val F1: {best_f1*100:.2f}%")

Training started!

Epoch 1/3


Using `past_key_values` as a tuple is deprecated and will be removed in v4.45. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Train Loss: 6.5333
Val Macro F1: 79.07%
 Best model saved!

Epoch 2/3
Train Loss: 0.4794
Val Macro F1: 86.26%
 Best model saved!

Epoch 3/3
Train Loss: 0.3916
Val Macro F1: 87.31%
 Best model saved!

Best Val F1: 87.31%


In [6]:
test_preds, test_true = evaluate(model, test_loader)

print("=== BLOOM LoRA Test Results ===")
print(classification_report(
    test_true, test_preds,
    target_names=['authentic', 'fake', 'ai_fake']
))

test_f1 = f1_score(test_true, test_preds, average='macro')
print(f"Test Macro F1: {test_f1*100:.2f}%")

=== BLOOM LoRA Test Results ===
              precision    recall  f1-score   support

   authentic       0.80      0.84      0.82       750
        fake       0.82      0.78      0.80       750
     ai_fake       0.98      0.98      0.98       750

    accuracy                           0.87      2250
   macro avg       0.87      0.87      0.86      2250
weighted avg       0.87      0.87      0.86      2250

Test Macro F1: 86.50%
